# Reusable Multi-File Data-Cleaning Template (R / tidyverse)

Copy this notebook for any new project that arrives as many similar CSVs.

**Replace the placeholders** (`YOUR_PATTERN`, column names, symbols, separators) and run top-to-bottom.

![Generic flowchart](us_census_cleaning_flowchart.png)

In [ ]:
library(readr)
library(dplyr)
library(tidyr)
library(stringr)
library(ggplot2)

## 1. Configuration (edit these values)

In [ ]:
DATA_DIR       <- "data"                       # folder containing the files
FILE_PATTERN   <- "states_.*\\.csv"             # regex for the files you want
JUNK_COL_POS   <- 1                            # position of pure index column to drop (or NULL)
PERCENT_COLS   <- c("Hispanic","White","Black","Native","Asian","Pacific")  # columns that contain %
CURRENCY_COLS  <- c("Income")                   # columns that contain $
SPLIT_COL      <- "GenderPop"                  # column to separate
SPLIT_INTO     <- c("male_pop", "female_pop")   # new names after split
SPLIT_SEP      <- "_"                          # separator character
STRIP_CHARS    <- c("M", "F")                   # characters to remove after split
SCALE_TO_UNIT  <- TRUE                         # divide percent cols by 100?
OUTPUT_CSV     <- "data/clean_output.csv"       # where to write the final table

## 2. Load & bind

In [ ]:
files <- list.files(path = DATA_DIR, pattern = FILE_PATTERN, full.names = TRUE)
stopifnot(length(files) > 0)
message("Found ", length(files), " files")

df_list <- lapply(files, read_csv, show_col_types = FALSE)
raw <- bind_rows(df_list)
message("Raw shape: ", paste(dim(raw), collapse = " x "))

## 3. Drop junk column(s)

In [ ]:
if (!is.null(JUNK_COL_POS)) {
  raw <- raw %>% select(-all_of(JUNK_COL_POS))
}

## 4. String cleaning

In [ ]:
# Remove % symbols
if (length(PERCENT_COLS) > 0) {
  raw <- raw %>% mutate(across(all_of(PERCENT_COLS), ~ str_remove_all(.x, "%")))
}

# Remove $ (or other currency) symbols
if (length(CURRENCY_COLS) > 0) {
  raw <- raw %>% mutate(across(all_of(CURRENCY_COLS), ~ str_remove_all(.x, "[\\$,]")))
}

## 5. Separate multi-value column

In [ ]:
if (!is.null(SPLIT_COL) && SPLIT_COL %in% names(raw)) {
  raw <- raw %>% separate(!!sym(SPLIT_COL), into = SPLIT_INTO, sep = SPLIT_SEP)
  # strip trailing characters if needed
  for (i in seq_along(SPLIT_INTO)) {
    ch <- STRIP_CHARS[i]
    if (!is.na(ch)) {
      raw <- raw %>% mutate(!!SPLIT_INTO[i] := str_remove(!!sym(SPLIT_INTO[i]), ch))
    }
  }
}

## 6. Type conversion + optional scaling

In [ ]:
to_numeric <- c(PERCENT_COLS, CURRENCY_COLS, SPLIT_INTO)
to_numeric <- intersect(to_numeric, names(raw))

raw <- raw %>% mutate(across(all_of(to_numeric), as.numeric))

if (SCALE_TO_UNIT && length(PERCENT_COLS) > 0) {
  raw <- raw %>% mutate(across(all_of(PERCENT_COLS), ~ .x / 100))
}

## 7. Deduplicate

In [ ]:
n_before <- nrow(raw)
raw <- raw %>% distinct()
message("Removed ", n_before - nrow(raw), " duplicate rows. Final shape: ", paste(dim(raw), collapse = " x "))

## 8. Persist & quick summary

In [ ]:
write_csv(raw, OUTPUT_CSV)
message("Clean data written to ", OUTPUT_CSV)

summary(raw)
head(raw)

## 9. Optional simulation hook

Re-run the pipeline under controlled missing-file or noise scenarios by wrapping the steps above in a function and calling `replicate()`.